In [0]:
# Importar librerías requeridas
import mlflow
import os
from datetime import datetime

In [0]:
# # Crear widgets de configuración
# # Core Configuration
# dbutils.widgets.text("catalog_name", "bluetab", "Catalog Name")
# dbutils.widgets.text("schema_name", "rag", "Schema Name")
# dbutils.widgets.text("environment", "dev", "Environment (dev/test/prod)")

# # Table Configuration
# dbutils.widgets.text("docs_text_table", "docs_text", "Documents Text Table")
# dbutils.widgets.text("docs_track_table", "docs_track", "Documents Tracking Table")
# dbutils.widgets.text("embeddings_table", "docs_text_embeddings", "Embeddings Table")

# # Volume Configuration
# dbutils.widgets.text("pdf_volume_path", "/Volumes/bluetab/rag/pdf_vol", "PDF Volume Path")

# # Model Configuration
# dbutils.widgets.text("embedding_model_name", "simple_embedding_model_bluetab", "Embedding Model Name")
# dbutils.widgets.text("llm_model_name", "flan_t5_base_model", "LLM Model Name")
# dbutils.widgets.text("rag_model_name", "appliance_chatbot_model", "RAG Model Name")

# # Endpoint Configuration
# dbutils.widgets.text("embedding_endpoint", "simple_embedding", "Embedding Endpoint")
# dbutils.widgets.text("llm_endpoint", "flan_t5_base_model", "LLM Endpoint")
# dbutils.widgets.text("rag_endpoint", "appliance_chatbot", "RAG Endpoint")
# dbutils.widgets.text("vector_search_endpoint", "doc_vector_endpoint", "Vector Search Endpoint")

# # Processing Configuration
# dbutils.widgets.text("chunk_size", "1000", "Text Chunk Size")
# dbutils.widgets.text("chunk_overlap", "200", "Text Chunk Overlap")
# dbutils.widgets.text("batch_size", "32", "Processing Batch Size")

# # Index configuration
# dbutils.widgets.text("index_name", "doc_idx", "Index name")

# # MLflow Configuration
# dbutils.widgets.text("experiment_name", "/Shared/RAG_Databricks_Bluetab_Pipeline", "MLflow Experiment Name")

# # Embedding model configuration
# dbutils.widgets.text("embedding_model_base", "distilbert-base-uncased", "Base Embedding Model")
# dbutils.widgets.text("embedding_dim", "768", "Embedding Dimensions")
# dbutils.widgets.text("max_length", "512", "Max Sequence Length")
# dbutils.widgets.dropdown("model_framework", "transformers", ["transformers", "sentence-transformers"], "Model Framework")

# # Endpoint configuration
# dbutils.widgets.dropdown("workload_size", "Small", ["Small", "Medium", "Large"], "Workload Size")
# dbutils.widgets.dropdown("scale_to_zero", "true", ["true", "false"], "Enable Scale to Zero")
# dbutils.widgets.text("model_version", "latest", "Model Version (or 'latest')")
# dbutils.widgets.text("endpoint_suffix", "", "Endpoint Name Suffix (optional)")

# # MLflow run management
# dbutils.widgets.text("parent_run_id", "", "Parent Run ID")
# dbutils.widgets.text("current_run", "", "Current Run")

In [0]:
# # Obtener valores de los widgets
# CATALOG_NAME = dbutils.widgets.get("catalog_name")
# SCHEMA_NAME = dbutils.widgets.get("schema_name")
# ENVIRONMENT = dbutils.widgets.get("environment")

# # Table names
# DOCS_TEXT_TABLE = dbutils.widgets.get("docs_text_table")
# DOCS_TRACK_TABLE = dbutils.widgets.get("docs_track_table")
# EMBEDDINGS_TABLE = dbutils.widgets.get("embeddings_table")

# # Volume path
# PDF_VOLUME_PATH = dbutils.widgets.get("pdf_volume_path")

# # Model names
# EMBEDDING_MODEL_NAME = dbutils.widgets.get("embedding_model_name")
# LLM_MODEL_NAME = dbutils.widgets.get("llm_model_name")
# RAG_MODEL_NAME = dbutils.widgets.get("rag_model_name")

# # Endpoint names
# EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint")
# LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
# RAG_ENDPOINT = dbutils.widgets.get("rag_endpoint")
# VECTOR_SEARCH_ENDPOINT = dbutils.widgets.get("vector_search_endpoint")

# # Processing parameters
# CHUNK_SIZE = int(dbutils.widgets.get("chunk_size"))
# CHUNK_OVERLAP = int(dbutils.widgets.get("chunk_overlap"))
# BATCH_SIZE = int(dbutils.widgets.get("batch_size"))

# # Index configuration
# INDEX = dbutils.widgets.get("index_name")

# # MLflow configuration
# EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

# # Embedding model configuration
# EMBEDDING_MODEL_BASE = dbutils.widgets.get("embedding_model_base")
# EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))
# MAX_LENGTH = int(dbutils.widgets.get("max_length"))
# MODEL_FRAMEWORK = dbutils.widgets.get("model_framework")

# # Endpoint configuration
# WORKLOAD_SIZE = dbutils.widgets.get("workload_size")
# SCALE_TO_ZERO = dbutils.widgets.get("scale_to_zero").lower() == "true"
# MODEL_VERSION = dbutils.widgets.get("model_version")
# ENDPOINT_SUFFIX = dbutils.widgets.get("endpoint_suffix")

# # Variables globales para gestión de parent/child runs
# PARENT_RUN_ID = dbutils.widgets.get("parent_run_id") or None
# CURRENT_RUN = dbutils.widgets.get("current_run") or None

# # Construir nombres completos
# DOCS_TEXT_TABLE_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{DOCS_TEXT_TABLE}"
# DOCS_TRACK_TABLE_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{DOCS_TRACK_TABLE}"
# EMBEDDINGS_TABLE_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{EMBEDDINGS_TABLE}"

# EMBEDDING_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{EMBEDDING_MODEL_NAME}"
# LLM_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{LLM_MODEL_NAME}"
# RAG_MODEL_FULL = f"{CATALOG_NAME}.{SCHEMA_NAME}.{RAG_MODEL_NAME}"

# INDEX_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX}"

# print("¡Configuración cargada correctamente!")
# print(f"Environment: {ENVIRONMENT}")
# print(f"Catalog: {CATALOG_NAME}")
# print(f"Schema: {SCHEMA_NAME}")

In [0]:
# Configurar MLflow
mlflow.set_registry_uri("databricks-uc")

try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
        print(f"Creado nuevo experimento: {EXPERIMENT_NAME}")
    else:
        experiment_id = experiment.experiment_id
        print(f"Usando experimento existente: {EXPERIMENT_NAME}")
    mlflow.set_experiment(EXPERIMENT_NAME)
except Exception as e:
    print(f"Error configurando experimento: {e}")
    mlflow.set_experiment("/Shared/RAG_Default")

In [0]:


def cleanup_active_runs():
    """Parar todas las runs de MLflow activas"""
    try:
        active_runs = mlflow.search_runs(filter_string="status = 'RUNNING'")
        for run_id in active_runs['run_id']:
            mlflow.end_run(run_id)
        print("✅ Runs activas limpiadas")
    except Exception as e:
        print(f"⚠️ Error limpiando runs activas: {e}")

# Ejecutar limpieza
cleanup_active_runs()

In [0]:
# Funciones utilitarias

def log_step(step_name, status="started", details=None):
    """Log pipeline step information"""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    message = f"[{timestamp}] Step: {step_name} - Status: {status}"
    if details:
        message += f" - Details: {details}"
    print(message)
    # if mlflow.active_run():
    #     mlflow.log_param(f"step_{step_name}_status", status)
    #     if details:
    #         mlflow.log_param(f"step_{step_name}_details", str(details))

def start_parent_run(run_name=None):
    """
    Inicia una parent run para todo el pipeline RAG
    """
    global PARENT_RUN_ID, CURRENT_RUN
    
    if run_name is None:
        run_name = f"RAG_Pipeline_{ENVIRONMENT}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    try:
        # Asegurar que no hay runs activas
        cleanup_active_runs()
        
        # Iniciar parent run
        CURRENT_RUN = mlflow.start_run(run_name=run_name)
        PARENT_RUN_ID = CURRENT_RUN.info.run_id
        
        # Log parámetros globales del pipeline
        mlflow.log_param("pipeline_type", "RAG_Databricks_Bluetab")
        mlflow.log_param("environment", ENVIRONMENT)
        mlflow.log_param("catalog", CATALOG_NAME)
        mlflow.log_param("schema", SCHEMA_NAME)
        mlflow.log_param("timestamp", datetime.now().isoformat())
        mlflow.log_param("run_type", "parent")
        
        print(f"🚀 Parent run iniciada: {run_name}")
        print(f"📍 Run ID: {PARENT_RUN_ID}")
        
        return PARENT_RUN_ID
        
    except Exception as e:
        print(f"❌ Error iniciando parent run: {e}")
        raise e

def start_child_run(task_name, parent_run_id=None):
    """
    Inicia una child run para una tarea específica del pipeline
    
    Args:
        task_name: Nombre de la tarea (ej: "01_create_tables", "02_pdf_processing")
        parent_run_id: ID de la parent run (usa PARENT_RUN_ID si no se especifica)
    """
    global CURRENT_RUN
    
    if parent_run_id is None:
        parent_run_id = PARENT_RUN_ID
    
    if parent_run_id is None:
        print("⚠️ No hay parent run activa. Iniciando parent run automáticamente...")
        start_parent_run()
        parent_run_id = PARENT_RUN_ID
    
    try:
        # Finalizar run actual si existe
        if CURRENT_RUN and CURRENT_RUN.info.run_id != parent_run_id:
            mlflow.end_run()
        
        # Iniciar child run
        run_name = f"{task_name}_{ENVIRONMENT}_{datetime.now().strftime('%H%M%S')}"
        CURRENT_RUN = mlflow.start_run(
            run_name=run_name,
            nested=True
        )
        
        # Log parámetros de la child run
        mlflow.log_param("task_name", task_name)
        mlflow.log_param("parent_run_id", parent_run_id)
        mlflow.log_param("environment", ENVIRONMENT)
        mlflow.log_param("run_type", "child")
        mlflow.log_param("timestamp", datetime.now().isoformat())
        
        print(f"📋 Child run iniciada: {run_name}")
        print(f"🔗 Parent run: {parent_run_id}")
        print(f"📍 Child run ID: {CURRENT_RUN.info.run_id}")
        
        log_step(task_name, "started", f"Child run iniciada para tarea {task_name}")
        
        return CURRENT_RUN.info.run_id
        
    except Exception as e:
        print(f"❌ Error iniciando child run: {e}")
        raise e

def end_child_run(status="success"):
    """
    Finaliza la child run actual
    """
    global CURRENT_RUN
    
    if CURRENT_RUN and CURRENT_RUN.info.run_id != PARENT_RUN_ID:
        try:
            mlflow.log_param("final_status", status)
            mlflow.end_run()
            print(f"✅ Child run finalizada con status: {status}")
            
            # Volver a la parent run
            if PARENT_RUN_ID:
                # Note: MLflow no permite reactivar runs, pero podemos trackear el estado
                CURRENT_RUN = None
                
        except Exception as e:
            print(f"⚠️ Error finalizando child run: {e}")

def end_parent_run(status="success"):
    """
    Finaliza la parent run del pipeline
    """
    global PARENT_RUN_ID, CURRENT_RUN
    
    try:
        # Finalizar child run si está activa
        if CURRENT_RUN and CURRENT_RUN.info.run_id != PARENT_RUN_ID:
            end_child_run(status)
        
        # Finalizar parent run si existe
        if PARENT_RUN_ID:
            # Reactivar parent run para logs finales
            try:
                with mlflow.start_run(run_id=PARENT_RUN_ID):
                    mlflow.log_param("pipeline_final_status", status)
                    mlflow.log_param("pipeline_end_time", datetime.now().isoformat())
                    print(f"🏁 Parent run finalizada con status: {status}")
            except:
                # Si no se puede reactivar, solo notificar
                print(f"🏁 Pipeline finalizado con status: {status}")
        
        # Limpiar variables globales
        PARENT_RUN_ID = None
        CURRENT_RUN = None
        
    except Exception as e:
        print(f"⚠️ Error finalizando parent run: {e}")

def get_current_run_info():
    """
    Obtiene información de las runs actuales
    """
    info = {
        "parent_run_id": PARENT_RUN_ID,
        "current_run_id": CURRENT_RUN.info.run_id if CURRENT_RUN else None,
        "is_parent_active": PARENT_RUN_ID is not None,
        "is_child_active": CURRENT_RUN is not None and (CURRENT_RUN.info.run_id != PARENT_RUN_ID if PARENT_RUN_ID else True)
    }
    return info

def create_table_if_not_exists(table_name, schema_sql):
    """Create table if it doesn't exist"""
    try:
        spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} {schema_sql}")
        log_step("create_table", "success", f"Table {table_name} created/verified")
        return True
    except Exception as e:
        log_step("create_table", "failed", f"Error creating {table_name}: {e}")
        return False

def get_databricks_host():
    """Get Databricks workspace URL"""
    try:
        return spark.conf.get("spark.databricks.workspaceUrl")
    except:
        return "https://dbc-ad7d5e59-0280.cloud.databricks.com/"

def build_endpoint_url(endpoint_name):
    """Build serving endpoint URL"""
    host = get_databricks_host()
    return f"https://{host}/serving-endpoints/{endpoint_name}/invocations"

In [0]:
# print("="*60)
# print("RAG DATABRICKS BLUETAB - CONFIGURATION SUMMARY")
# print("="*60)
# print(f"Environment: {ENVIRONMENT}")
# print(f"Catalog: {CATALOG_NAME}")
# print(f"Schema: {SCHEMA_NAME}")
# print()
# print("TABLES:")
# print(f"  - Documents Text: {DOCS_TEXT_TABLE_FULL}")
# print(f"  - Documents Track: {DOCS_TRACK_TABLE_FULL}")
# print(f"  - Embeddings: {EMBEDDINGS_TABLE_FULL}")
# print()
# print("MODELS:")
# print(f"  - Embedding: {EMBEDDING_MODEL_FULL}")
# print(f"  - LLM: {LLM_MODEL_FULL}")
# print(f"  - RAG: {RAG_MODEL_FULL}")
# print()
# print("ENDPOINTS:")
# print(f"  - Embedding: {EMBEDDING_ENDPOINT}")
# print(f"  - LLM: {LLM_ENDPOINT}")
# print(f"  - RAG: {RAG_ENDPOINT}")
# print(f"  - Vector Search: {VECTOR_SEARCH_ENDPOINT}")
# print()
# print("PROCESSING:")
# print(f"  - Chunk Size: {CHUNK_SIZE}")
# print(f"  - Chunk Overlap: {CHUNK_OVERLAP}")
# print(f"  - Batch Size: {BATCH_SIZE}")
# print()
# print("MLFLOW:")
# print(f"  - Experiment: {EXPERIMENT_NAME}")
# print()
# print("RUN MANAGEMENT:")
# run_info = get_current_run_info()
# print(f"  - Parent Run ID: {run_info['parent_run_id'] or 'None'}")
# print(f"  - Current Run ID: {run_info['current_run_id'] or 'None'}")
# print(f"  - Parent Active: {'✅' if run_info['is_parent_active'] else '❌'}")
# print(f"  - Child Active: {'✅' if run_info['is_child_active'] else '❌'}")
# print()
# print("AVAILABLE FUNCTIONS:")
# print("  - start_parent_run(run_name=None)")
# print("  - start_child_run(task_name, parent_run_id=None)")
# print("  - end_child_run(status='success')")
# print("  - end_parent_run(status='success')")
# print("  - get_current_run_info()")
# print("="*60)